In [ ]:
### Version 2 Morges with my own manning values 

In [2]:
import sys
sys.path.append('/storage/homefs/ge24z347/Campgrounds/src')
from DEM_processing import rename_file_extension

In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import os

# -------------------------------------
# Define file paths
# -------------------------------------
dem_path = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2.tif"
manning_path = "/rs_scratch/users/ge24z347/Data_forprocess/Arealstatistik_processing/Morges_2m_manning.tif"
output_tif = "/rs_scratch/users/ge24z347/Data_forprocess/Arealstatistik_processing/Morges_2m_v2.tif"
output_asc = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2.asc"

# -------------------------------------
# Function to rename .asc to .n
# -------------------------------------
def rename_file_extension(filepath, new_ext):
    base = os.path.splitext(filepath)[0]
    new_path = base + new_ext
    os.rename(filepath, new_path)
    return new_path

# -------------------------------------
# Load DEM (reference grid)
# -------------------------------------
with rasterio.open(dem_path) as dem:
    dem_profile = dem.profile
    dem_transform = dem.transform
    dem_crs = dem.crs
    width, height = dem.width, dem.height

# -------------------------------------
# Load and reproject Manning raster if needed
# -------------------------------------
with rasterio.open(manning_path) as src:
    if src.crs != dem_crs or src.transform != dem_transform:
        data = np.empty((1, height, width), dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=data[0],
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dem_transform,
            dst_crs=dem_crs,
            resampling=Resampling.nearest  # no interpolation
        )
    else:
        data = src.read()

# -------------------------------------
# Replace NaNs with -9999
# -------------------------------------
nodata_value = -9999
manning_data_filled = np.where(np.isnan(data[0]), nodata_value, data[0])

# -------------------------------------
# Save reprojected GeoTIFF
# -------------------------------------
profile = dem_profile.copy()
profile.update(dtype=rasterio.float32, count=1, nodata=nodata_value)

with rasterio.open(output_tif, 'w', **profile) as dst:
    dst.write(manning_data_filled.astype(np.float32), 1)

# -------------------------------------
# Export to ASCII with controlled precision (.asc)
# -------------------------------------
transform = dem_transform
ncols = width
nrows = height
xllcorner = transform.c
yllcorner = transform.f + transform.e * nrows  # y origin at top
cellsize = transform.a

with open(output_asc, 'w') as f:
    f.write(f"ncols         {ncols}\n")
    f.write(f"nrows         {nrows}\n")
    f.write(f"xllcorner     {xllcorner:.6f}\n")
    f.write(f"yllcorner     {yllcorner:.6f}\n")
    f.write(f"cellsize      {cellsize:.6f}\n")
    f.write(f"NODATA_value  {nodata_value}\n")
    np.savetxt(f, manning_data_filled, fmt="%.3f")

# -------------------------------------
# Rename to .n
# -------------------------------------
manning_n_path = rename_file_extension(output_asc, ".n")

print(f" Manning raster saved:\n - GeoTIFF: {output_tif}\n - ASCII: {manning_n_path}")

 Manning raster saved:
 - GeoTIFF: /rs_scratch/users/ge24z347/Data_forprocess/Arealstatistik_processing/Morges_2m_v2.tif
 - ASCII: /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2.n


In [ ]:
## stage file

In [3]:
from lisflood_inputdata import create_stage_file
catchment_location_csv = "/rs_scratch/users/ge24z347/Data_forprocess/catchment_location.csv"
selected_id= 13
output_stage_file= "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2.stage"
# Call the function
create_stage_file(catchment_location_csv, selected_id, output_stage_file)

.stage file created successfully at /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2.stage


In [ ]:
## create the netcdfile 

In [3]:
from lisflood_inputdata import create_precip_automatic_camp_v2
base_name = "Morges_2m_v2"
dem_file_path = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2.tif"  # 
output_dir = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/"
buffer_start = 5  # minimum rainfall in mm
buffer_end = 75    # maximum rainfall in mm
buffer_step = 10   # step (e.g., 10 mm → 10, 20, 30 mm)
create_precip_automatic_camp_v2(base_name, dem_file_path, buffer_start, buffer_end, buffer_step, output_dir)

Creating NetCDF for Morges_2m_v2 with total precipitation = 5 mm
TUFLOW-compatible NetCDF file created: /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_5.nc
Created NetCDF: /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_5.nc
Creating NetCDF for Morges_2m_v2 with total precipitation = 15 mm
TUFLOW-compatible NetCDF file created: /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_15.nc
Created NetCDF: /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_15.nc
Creating NetCDF for Morges_2m_v2 with total precipitation = 25 mm
TUFLOW-compatible NetCDF file created: /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_25.nc
Created NetCDF: /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_25.nc
Creating NetCDF for Morges_2m_v2 with total precipitation = 35 mm
TUFLOW-compatible NetCDF file created: /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/M

In [ ]:
## par file 


In [4]:
from DEM_processing import create_par_file_90min
import os

base_name = "Morges_2m_v2"
output_dir = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2"

for total_precip in range(5, 76, 10):  # 5, 15, ..., 75
    output_file = os.path.join(output_dir, f"{base_name}_{total_precip}.par")
    create_par_file_90min(base_name, total_precip, output_file)

✅ .par file created at /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_5.par
✅ .par file created at /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_15.par
✅ .par file created at /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_25.par
✅ .par file created at /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_35.par
✅ .par file created at /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_45.par
✅ .par file created at /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_55.par
✅ .par file created at /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_65.par
✅ .par file created at /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2_75.par


In [5]:
### rename the DEM File 
from DEM_processing import convert_tif_to_asc
dem_tif="/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2.tif"
output_asc="/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2.asc"
convert_tif_to_asc (dem_tif, output_asc, desired_nodata_value=-9999)
dem_n_path = rename_file_extension("/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2.asc", ".dem")

Metadata:
ncols: 1156
nrows: 1543
xllcorner: 2525738.0
yllcorner: 1150016.0
cellsize: 2.0
NODATA_value: -9999
CRS saved to: /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2.prj
Conversion complete. ASCII file saved to /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2.asc
 File renamed to: /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_v2/Morges_2m_v2.dem
